# Defect localization

**Recipe 81** · Level 3 (Intermediate) · Decision type: `Box` + `Noul`

Check for one specified visible scratch, dent, or broken component, then draw and evaluate its bounding rectangle while suppressing boxes for absent targets.

Sources:

- S02: [TypeSafe AI: Primitives (Questions)](https://docs.typesafe.ai/primitives)
- S03: [TypeSafe AI: Confidence](https://docs.typesafe.ai/confidence)
- S09: [TypeSafe AI: Images](https://docs.typesafe.ai/images)
- S10: [TypeSafe AI: Box](https://docs.typesafe.ai/primitives/box)
- S11: [TypeSafe AI: Models and long context](https://docs.typesafe.ai/models)
- S12: [TypeSafe AI: Jev 1.14 release](https://typesafe.ai/blog/jev-1-14)

## What you will build

Locate one requested scratch, dent, or fractured terminal on a service panel, draw its rectangle, and inspect an enlarged crop. These are original synthetic drawings, not photographs. Three independent `Noul` propositions establish presence, uniqueness, and visibility before a dependent `Box` request. A `Box` always returns coordinates, including for absent targets ([S10](https://docs.typesafe.ai/primitives/box)); its existence alone never establishes a defect.

Python decides whether to annotate, report absence, or request [review](../../docs/glossary.md#review)—a person must decide the same unresolved inspection. Proposed annotations are simulated; no equipment status or maintenance record changes.

## Setup and run mode

There are 30 drawings: 14 validation examples, 14 held-out test examples, and two unscored demonstrations. Answers have synthetic [provenance](../../docs/glossary.md#provenance), meaning they were authored rather than returned by inference. Some are deliberately wrong. The images include different defect kinds, printed marks, glare, partial covers, and more than one candidate.

Image requests use the experimental images flag and the pinned `jev-1.14.0` model ([S09](https://docs.typesafe.ai/images), [S11](https://docs.typesafe.ai/models), [S12](https://typesafe.ai/blog/jev-1-14)). Offline replay remains the default. The recipe imposes a 60-request cap; an optional live backend also counts attempts and retries against its own budget.

In [ ]:
from IPython.display import display
from matplotlib.figure import Figure
from PIL import Image

from jev_cookbook import get_backend, load_helpers
from jev_cookbook.fixtures import load_inputs, load_labels, responses_path
from jev_cookbook.style import INK, PINK, apply_style, run_header, show_answer
from jev_cookbook.vision import crop_box, draw_boxes

apply_style()
helpers = load_helpers()
examples, labels = load_inputs(), load_labels()
scored = [e for e in examples if e.split in ("validation", "test")]
backend = get_backend(fixtures=responses_path(), images=True, model="jev-1.14.0")
inspector = helpers.Inspector(backend, max_requests=60)
offline = backend.mode in ("synthetic", "scripted")
check = " (a pipeline check, not a Jev result)" if offline else ""
selection = " (selection step)"
run_header(81, "Defect localization", backend=backend, n_examples=len(scored))

## The state

Inference receives the panel pixels, the requested kind of defect, and a fixed description of the inspection window. Local filenames, split membership, authored geometry, and expected outcomes stay outside the request. Image references are opaque content hashes. The [replay key](../../docs/glossary.md#replay-key)—the identity used to find a stored response—binds the state, question definitions, and referenced image bytes.

In [ ]:
demo = next(e for e in examples if e.id == "d01")
state, questions, assets = helpers.request(demo.fields)
print(state)
with Image.open(demo.fields["image"]) as image:
    display(image.copy())

## The questions

A `Noul` answer is the probability that its proposition is true ([S02](https://docs.typesafe.ai/primitives)). Presence, uniqueness, and an unobstructed view are independent judgments and share one request. The acquisition screen requires each probability to be at least 0.80 before asking for a rectangle. This permissive screen is fixed before selection; the final policy may require stronger evidence.

The second request asks only for the extent of the requested defect. It uses `Box(instructions=...)`, with no criteria. A broken component includes its separated pieces; a scratch includes its ragged extent; a dent includes its rim. The same definitions serve replay and optional live calls.

In [ ]:
for name, question in questions.items():
    print(name, question.to_dict())
print("Dependent request:", helpers.build_questions(locate=True)["defect"].to_dict())

## One answer up close

Here is the typed answer before any aggregate. `Noul` exposes a yes probability, not a separate confidence field. `Box` supplies normalized edges and may omit confidence. The pink rectangle comes from that answer, and Python expands its crop by 20% on every side. This display crop is not another inference input.

In [ ]:
first, rectangle = inspector.acquire(demo.fields)
for name, answer in first.answers.items():
    print(name)
    show_answer(answer)
if rectangle is None:
    print("The acquisition screen did not authorize a Box request.")
else:
    show_answer(rectangle)
    with Image.open(demo.fields["image"]) as image:
        display(draw_boxes(image, {"scratch": rectangle}))
        crop = crop_box(image, rectangle, expand=0.2)
        display(crop.resize((480, round(480 * crop.height / crop.width))))

## Python's part

A [confidence gate](../../docs/glossary.md#confidence-gate) is a rule that defers an answer whose certainty is too low. Here the final policy gates the three `Noul` probabilities and the reported box confidence separately; their values are not interchangeable ([S03](https://docs.typesafe.ai/confidence)). Missing box confidence always requires review. Strong absence needs both low presence and low uniqueness, plus a clear view, and emits no rectangle.

A unique, visible target must pass the gates before Python records `propose_annotation` in the simulated `ActionLog`. Occlusion, multiple targets, contradictory presence and uniqueness, and weak evidence enter the `ReviewQueue`. The inspector caches each request, spends at most two requests per image, and stops at its explicit cap with no automatic retries. This cache lets validation try several policies without repeating inference.

In [ ]:
ambiguous = next(e for e in examples if e.id == "d02")
ambiguous_first, ambiguous_box = inspector.acquire(ambiguous.fields)
print("Presence / uniqueness / clear:", [a.noul for a in ambiguous_first.answers.values()])
print("Box requested:", ambiguous_box is not None)
with Image.open(ambiguous.fields["image"]) as image:
    display(image.copy())

## Evaluation

A [gold label](../../docs/glossary.md#gold-label) is the authored expected outcome and, for a uniquely visible defect, its rectangle. Labels and their reasons are readable in `fixtures/labels.jsonl`; no label enters inference or the baseline. A located answer is correct if its intersection-over-union (IoU: intersection area divided by union area) is at least 0.50. A fully visible absent target is correctly answered by `absent`. Multiple or partly covered candidates require review.

[Coverage](../../docs/glossary.md#coverage) is the fraction actually answered; [accuracy](../../docs/glossary.md#accuracy) is correctness among those answers; [risk](../../docs/glossary.md#risk) is one minus that accuracy. A review never counts as an answer, including a correct deferral. The shared `evaluate_outcomes` helper receives the rule's actual accepted flags.

Only validation selects the policy: try presence/uniqueness/visibility thresholds {0.80, 0.90, 0.95} crossed with box thresholds {0.60, 0.80, 0.90}. Maximize answered count subject to accuracy at least 0.90; ties prefer accuracy, then stricter presence, then stricter box confidence. Failure to meet the objective stops selection. Freeze both values before looking at test outcomes.

In [ ]:
validation = [e for e in examples if e.split == "validation"]
validation_observations = [inspector.acquire(e.fields) for e in validation]
policy, trials = helpers.select_policy(
    validation_observations,
    [labels[e.id] for e in validation],
    target_accuracy=0.90,
)
for candidate, scores in trials:
    print(
        f"Validation N={scores.n_total}: p={candidate.presence:.2f}, box={candidate.box:.2f}, "
        f"answered={scores.n_answered}, coverage={scores.coverage:.4f}, "
        f"accuracy={scores.accuracy:.4f}, risk={scores.risk:.4f}{selection}{check}"
    )
print(f"Frozen policy: {policy}; validation N={len(validation)}{selection}{check}")

### Pixel baseline and denominators

The deterministic baseline finds the largest connected dark region inside a fixed central window: grayscale below 105, at least 18 connected pixels. It always reports that rectangle or absence. This is a plausible quick surface-anomaly detector, but it does not understand the requested defect kind, shaded dents, fractures, or duplicate defects. Its constants are fixed in advance, and it reads only image pixels; neither filenames nor labels guide its prediction.

Both methods see the same test images. The primary geometry report includes **every uniquely visible or absent gold scene**, even when the policy defers. A reviewed present target has no delivered box, contributes zero IoU, and counts as a delivery miss. We split those misses into accepted absence errors and deferred present targets. A returned box with IoU below 0.50 is also counted separately as `poor_overlap`.

Multiple and occluded candidates have no single gold rectangle. They remain in coverage and accuracy, while boxes emitted on those scenes are reported separately as `invalid_scene_boxes`, not as false positives on genuinely absent targets. The resolved-only mean IoU is supplemental and prints its smaller present-target denominator. This makes the price of review visible in the primary comparison.

In [ ]:
test = [e for e in examples if e.split == "test"]
test_labels = [labels[e.id] for e in test]
outcomes = [inspector.inspect(e.id, e.fields, policy) for e in test]
baseline = [helpers.baseline_for(e.fields) for e in test]
reports = {}
if backend.mode == "live":
    print("Unrecorded live exploration: no reportable held-out metrics. Record first.")
else:
    reports = {
        "Gated inspection": helpers.summarize(outcomes, test_labels),
        "Dark-region baseline": helpers.summarize(baseline, test_labels),
    }
    for name, report in reports.items():
        print(f"Test N={len(test)}: {name}{check}")
        print(
            f"answered={report['answered']}, review={report['review']}, "
            f"coverage={report['coverage']:.4f}, accuracy={report['accuracy']:.4f}, "
            f"risk={report['risk']:.4f}{check}"
        )
        print(
            f"Geometry N={report['geometry_n']}, present={report['present']}, absent={report['absent']}, "
            f"all-present mean IoU={report['mean_iou']:.4f}, delivery misses={report['missed']}, "
            f"false positives={report['false_positives']}, poor overlap={report['poor_overlap']}{check}"
        )
        print(
            f"Miss breakdown: accepted absence={report['accepted_misses']}, "
            f"deferred present={report['reviewed_present']}; "
            f"boxes on multiple/occluded scenes={report['invalid_scene_boxes']}{check}"
        )
        print(
            f"Supplemental resolved present N={report['resolved_present']}, "
            f"resolved mean IoU={report['resolved_mean_iou']:.4f}{check}"
        )

In [ ]:
if reports:
    values = [report["coverage"] for report in reports.values()]
    print(
        f"Test N={len(test)} plotted coverage: "
        f"{dict(zip(reports, [round(v, 4) for v in values], strict=True))}{check}"
    )
    figure = Figure(figsize=(7, 3.1))
    ax = figure.subplots()
    ax.barh(list(reports), values, color=[PINK, INK])
    ax.set_xlim(0, 1)
    ax.set_xlabel("Fraction answered (reviews excluded)")
    ax.set_title(f"Test N={len(test)}{check}", fontsize=10)
    figure.tight_layout()
    display(figure)

### Accepted mistakes remain visible

The fixtures intentionally include confidently wrong held-out answers. The display below uses already computed outcomes, with no second test run and no policy changes. A reflection can be accepted as a scratch, a dent rectangle can land on clean metal, and a fractured component can be missed as absent. None of the gates certifies correctness.

In [ ]:
if backend.mode != "live":
    wrong = [
        (e, outcome)
        for e, outcome, label in zip(test, outcomes, test_labels, strict=True)
        if outcome.accepted and not helpers.correct(outcome, label)
    ]
    print(f"Accepted errors among test N={len(test)}: {[e.id for e, _ in wrong]}{check}")
    for example, outcome in wrong:
        print(example.id, outcome.outcome, labels[example.id]["note"])
        boxes = {"accepted": outcome.box} if outcome.box is not None else {}
        gold_box = labels[example.id]["box"]
        if gold_box is not None:
            boxes["gold"] = gold_box
        with Image.open(example.fields["image"]) as image:
            display(draw_boxes(image, boxes))
print("Simulated annotation proposals:", inspector.actions.to_dicts())
print("Pending review items:", inspector.reviews.to_dicts())

## What was and was not measured

Not measured live: the offline run replays synthetic answers, so its numbers check the pipeline and say nothing about how Jev performs. The header and the cell below disclose the mode, N, and, for recorded inference, the returned model and capture dates. Validation N=14 is selection; test N=14 is the held-out report; two demonstrations are unscored. One request batches three propositions, so request counts are not question counts.

The dataset's simple drawings do not represent real inspection imagery. The rule can accept a confident mistake. The comparison is between two complete pipelines with different review rates, not a like-for-like estimate of model quality. Tokens, actual inference latency, and cost remain `None` when unmeasured. Replay runtime is never used as inference latency.

For an optional recording, `helpers.record_corpus` uses the same request triples and the shared `record` tool, and decides the second stage from actual recorded first-stage answers. Capture to a new responses file, the helper writes input replay-key lists from the actual recorded paths; do not mix synthetic and recorded answers. The recorder keeps returned model, date and reported usage. No live call is made by this notebook's offline executor.

In [ ]:
if offline:
    print(f"Provenance: {backend.mode}. Not measured live: a pipeline check, not a Jev result.")
    print(
        f"N: {len(scored)} scored examples; validation N={len(validation)}, test N={len(test)}{check}"
    )
else:
    dates = ", ".join(getattr(backend, "recorded_dates", ()) or ()) or "this unrecorded run"
    print(f"Provenance: {backend.mode}, returned model {backend.model}; captured {dates}")
    print(f"N: {len(scored)} scored examples; validation N={len(validation)}, test N={len(test)}")
print(f"Unique requests: {inspector.calls}; cap: {inspector.max_requests}{check}")
print("First request accounting:", inspector.accounting[0])
print("Unmeasured fields remain None; no replay timings or inferred charges.")

## Next steps

Add independently labeled real panel photographs and a new held-out split before measuring live behavior. Include dents under several lighting directions and scratches beside printed traces; use the same frozen policy and report both review rates and localization errors. A production inspection study would need a richer labeling protocol for occlusion and multiple defects.

Continue with [tool outcome verification](../29-tool-outcome-verification/) for exact action boundaries or [sentiment classification](../01-sentiment-classification/) for a smaller validation/test lesson. The shared [vision helpers](../../docs/vision.md) explain crop geometry and image identity.